# Culture Survey 2012 — SPSS Parsing & Exploration

This notebook loads and explores the **Kultura 2012** survey data from three Czech regions (Ústecký, Vysočina, Zlínský).

## Quick SPSS → Pandas Refresher

- **SPSS `.sav` files** store data + metadata (variable labels, value labels, missing value codes)
- **`pyreadstat`** reads `.sav` files and returns:
  - A pandas DataFrame (the actual data)
  - A metadata object (labels, creation info, etc.)
- Once loaded, you work with the DataFrame exactly like any pandas DataFrame

### Key concepts:
- **Variable labels**: Human-readable descriptions of columns (e.g., `Q1_1` → "How often do you visit libraries?")
- **Value labels**: Categorical codes mapped to text (e.g., `1` = "Never", `2` = "Once a month", `3` = "Weekly")
- **Missing values**: SPSS can define custom missing codes (e.g., `-99`, `999`) that pyreadstat converts to `NaN`

## 1. Setup — Load the Data

In [1]:
import pyreadstat
import pandas as pd
from pathlib import Path

In [2]:
# Path to the SPSS .sav file (relative to notebook location)
SPSS_FILE = Path("../data/culture_dataverse_files/Kult2012_3kraje_UstVysZli_CSDA_pub_nove_bez_jmen.sav")

# Read SPSS file — returns tuple: (DataFrame with data, metadata object with labels)
df, meta = pyreadstat.read_sav(SPSS_FILE)

# Print basic info: number of rows × columns
print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
# Print the file label from metadata (if any)
print(f"\nFile label: {meta.file_label or '(not set)'}")
# Print the table name from metadata
print(f"Table name: {meta.table_name}")
# Print the creation timestamp of the SPSS file
print(f"Created: {meta.creation_time}")

Loaded 3679 rows × 275 columns

File label: (not set)
Table name: None
Created: 2015-01-07 11:54:08


## 2. Explore Variable Labels

SPSS stores human-readable labels for each variable. Let's see what we're working with.

In [8]:
# All variable labels as a DataFrame for easy browsing
var_labels_df = pd.DataFrame({
    'variable': list(meta.column_names_to_labels.keys()),
    'label': list(meta.column_names_to_labels.values())
})

print(f"Total variables: {len(var_labels_df)}\n")
var_labels_df.head(20)

Total variables: 275



,variable,label
0,CD,CD Číslo dotazníku
1,KRAJ,Kraj
2,VER,VER Verze dotazníku - kraj
3,KOD,KOD Číslo kodéra
4,SUP,SUP Číslo superkontrolora
5,POR,POR Číslo pořizovače
6,K.1a,"K.1a Odkud čerpá informace o minulosti, tradic..."
7,K.1b,"K.1b Odkud čerpá informace o minulosti, tradic..."
8,K.1c,"K.1c Odkud čerpá informace o minulosti, tradic..."
9,K.2,"K.2 Vypráví, zprostředkovává ostatním minulost..."


## 3. Explore Value Labels (Categorical Variables)

SPSS stores categories as numeric codes with text labels. Let's see which variables have them.

In [9]:
# Show all value label sets
print(f"Variables with value labels: {len(meta.value_labels)}\n")

# Pick a few examples to inspect
for var_name in list(meta.value_labels.keys())[:5]:
    print(f"\n{var_name}:")
    for code, label in meta.value_labels[var_name].items():
        print(f"  {code} → {label}")

Variables with value labels: 110


labels0:
  1.0 → Ústecký kraj
  2.0 → Kraj Vysočina
  3.0 → Zlínský kraj

labels1:
  1.0 → Ústecký kraj
  2.0 → Kraj Vysočina
  3.0 → Zlínský kraj

labels2:
  0.0 → BEZ ODPOVĚDI
  1.0 → Ze školy
  2.0 → Od rodinných příslušníků
  3.0 → Od známých a dalších pamětníků
  4.0 → Z návštěv výstav, muzeí, folklórních festivalů a podobných akcí
  5.0 → Z knih, dokumentů
  6.0 → Z internetu
  7.0 → Z médií (televize, rozhlas, noviny)
  8.0 → Odjinud
  98.0 → O HISTORII A TRADICE REGIONU SE NEZAJÍMÁ
  99.0 → NEVÍ

labels3:
  0.0 → BEZ ODPOVĚDI
  1.0 → velmi často
  2.0 → často
  3.0 → občas
  4.0 → zřídka
  5.0 → nikdy
  9.0 → NEVÍ

labels4:
  0.0 → BEZ ODPOVĚDI
  1.0 → zná, ale neumí
  2.0 → zná, umí je
  3.0 → zná, aktivně se věnuje
  4.0 → nezná žádné
  5.0 → nic takého u nich není
  9.0 → NEVÍ


## 4. Demographic Variables — First Look

Let's identify key demographic columns and their distributions.

In [10]:
# Common demographic variable patterns in this dataset
demo_vars = [v for v in df.columns if any(k in v.lower() for k in ['vek', 'pohl', 'vzdel', 'kraj', 'obec'])]

print("Potential demographic variables:")
for var in demo_vars:
    label = meta.column_names_to_labels.get(var, "(no label)")
    print(f"  {var}: {label}")

Potential demographic variables:
  KRAJ: Kraj
  t_vek_4: t_vek_4 - transformovaný věk - 4 kategorie
  kodkraj: Kód kraje
  nazevkraj: Název kraje
  pohlavi: Pohlaví
  vek: Věk
  vek6a: Věk 6 kat.
  vek4: Věk 4 kat. (kvóta)


In [11]:
# Show value counts for categorical variables with labels
def labeled_value_counts(series, var_name):
    """Print value counts with SPSS labels."""
    counts = series.value_counts().sort_index()
    
    # Get the value label set name for this variable
    label_set_name = meta.variable_to_label.get(var_name)
    labels = meta.value_labels.get(label_set_name, {}) if label_set_name else {}
    
    print(f"\n{var_name}: {meta.column_names_to_labels.get(var_name, '(no label)')}")
    print("-" * 50)
    for val, count in counts.items():
        label = labels.get(val, str(val)) if labels else str(val)
        print(f"  {val:>4} ({label:<20}): {count:>4} ({100*count/len(series):5.1f}%)")

# Example: check first few categorical variables
for var in list(meta.column_names_to_labels.keys())[:3]:
    labeled_value_counts(df[var], var)


CD: CD Číslo dotazníku
--------------------------------------------------
   1.0 (1.0                 ):    1 (  0.0%)
   2.0 (2.0                 ):    1 (  0.0%)
   3.0 (3.0                 ):    1 (  0.0%)
   4.0 (4.0                 ):    1 (  0.0%)
   5.0 (5.0                 ):    1 (  0.0%)
   6.0 (6.0                 ):    1 (  0.0%)
   7.0 (7.0                 ):    1 (  0.0%)
   8.0 (8.0                 ):    1 (  0.0%)
   9.0 (9.0                 ):    1 (  0.0%)
  10.0 (10.0                ):    1 (  0.0%)
  11.0 (11.0                ):    1 (  0.0%)
  12.0 (12.0                ):    1 (  0.0%)
  13.0 (13.0                ):    1 (  0.0%)
  14.0 (14.0                ):    1 (  0.0%)
  15.0 (15.0                ):    1 (  0.0%)
  16.0 (16.0                ):    1 (  0.0%)
  17.0 (17.0                ):    1 (  0.0%)
  18.0 (18.0                ):    1 (  0.0%)
  19.0 (19.0                ):    1 (  0.0%)
  20.0 (20.0                ):    1 (  0.0%)
  21.0 (21.0             

## 5. Missing Values

SPSS often defines custom missing codes. pyreadstat converts these to `NaN` by default.

In [12]:
# Missing value summary
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_pct = 100 * missing_summary / len(df)

print("Top 15 variables by missing values:\n")
result = pd.DataFrame({'missing_count': missing_summary.head(15), 'missing_pct': missing_pct.head(15)})
result[result['missing_count'] > 0]

Top 15 variables by missing values:



,missing_count,missing_pct
xxxxx1,3679,100.000000
K.79c,3537,96.140256
K.18b,3293,89.508018
K.79b,3034,82.468062
K.20,3021,82.114705
K.19,3016,81.978799
K.79a,2515,68.360968
K.7,2427,65.969013
K.81,2049,55.694482
K.71,1956,53.166621


## 6. Optional: Apply Value Labels Automatically

Convert coded values to their text labels for easier exploration.

In [13]:
# Re-read with value labels applied as strings
df_labeled, _ = pyreadstat.read_sav(SPSS_FILE, apply_value_formats=True)

# Compare: original vs labeled
print("Original (coded):")
print(df.iloc[:3, :5])

print("\nLabeled (text values):")
print(df_labeled.iloc[:3, :5])

Original (coded):
       CD  KRAJ  VER   KOD   SUP
0  1158.0   1.0  1.0  13.0  11.0
1  1159.0   1.0  1.0  13.0  11.0
2  1160.0   1.0  1.0  13.0  11.0

Labeled (text values):
       CD          KRAJ           VER   KOD   SUP
0  1158.0  Ústecký kraj  Ústecký kraj  13.0  11.0
1  1159.0  Ústecký kraj  Ústecký kraj  13.0  11.0
2  1160.0  Ústecký kraj  Ústecký kraj  13.0  11.0


In [14]:
df_labeled.head(5)

,CD,KRAJ,VER,KOD,SUP,POR,K.1a,K.1b,K.1c,K.2,...,velobce3b,pohlavi,t_vzd,vek,vek6a,vek4,prijemDom17k,prijemDom3ter,k71vstupenky,k72knihyCD
0,1158.0,Ústecký kraj,Ústecký kraj,13.0,11.0,11.0,"Z médií (televize, rozhlas, noviny)",NaN,NaN,zřídka,...,do 2 tis.,žena,střední bez maturity,57.0,46-59,45-62,27001 - 29000,II. tercil,100.0,0.0
1,1159.0,Ústecký kraj,Ústecký kraj,13.0,11.0,11.0,Od známých a dalších pamětníků,Z internetu,"Z knih, dokumentů",zřídka,...,do 2 tis.,žena,střední bez maturity,38.0,36-45,30-44,29001 - 30000,II. tercil,300.0,0.0
2,1160.0,Ústecký kraj,Ústecký kraj,13.0,11.0,11.0,Ze školy,Z internetu,"Z médií (televize, rozhlas, noviny)",nikdy,...,do 2 tis.,žena,střední s maturitou,21.0,20-26,20-29,29001 - 30000,II. tercil,600.0,50.0
3,1249.0,Ústecký kraj,Ústecký kraj,8.0,2.0,2.0,Z internetu,Od známých a dalších pamětníků,"Z knih, dokumentů",zřídka,...,do 2 tis.,muž,střední bez maturity,49.0,46-59,45-62,36001 - 40000,III. tercil,1500.0,0.0
4,1251.0,Ústecký kraj,Ústecký kraj,8.0,2.0,2.0,"Z knih, dokumentů",Od známých a dalších pamětníků,NaN,občas,...,do 2 tis.,žena,střední bez maturity,60.0,60-69,45-62,29001 - 30000,II. tercil,1200.0,200.0


## Next Steps

- Identify specific research questions (e.g., cultural participation by region/education)
- Cross-tabulate variables using `pd.crosstab()`
- Group by region (`KRAJ`) and compare responses
- Export cleaned data to CSV/Parquet for further analysis